# Estrategia de ofertas FDS - Pricing Justo

Notebook delgado: la logica de negocio (elegibilidad, score, cascada de
mecanica, guardrail economico, modelo economico, post-mortem) vive en
`backend/pricing/` como funciones reutilizables - este notebook solo
conecta a Snowflake, llama esas funciones y reporta resultados.

- `backend/pricing/catalog.py` - FULL_MASTER_CATALOG, VW_PRICING_DASHBOARD
- `backend/pricing/elasticity.py` - ATHENEA + fallback en cascada
- `backend/pricing/strategy.py` - toda la estrategia de ofertas
- `backend/pricing/postmortem.py` - vistas/tabla de medicion recurrente en SANDBOX

Ver el plan completo en `/home/alejomd17/.claude/plans/vamos-a-cambiar-de-glowing-plum.md`.

In [ ]:
import sys
sys.path.insert(0, "..")  # para que "backend" se resuelva desde src/

import pandas as pd

from backend.pricing import strategy, postmortem
from backend.pricing.snowflake_conn import get_connection

## Parametros

In [ ]:
RUTA_OPORTUNIDAD = "../data/inputs/oportunidad.xlsx"
RUTA_DESCUENTOS  = "../data/inputs/Descuentos comerciales.xlsx"

# Fin de semana objetivo (ajustar cada vez que se corra para un nuevo FDS)
WEEKEND_INICIO = pd.Timestamp("2026-07-03")   # viernes
WEEKEND_FIN    = pd.Timestamp("2026-07-05")   # domingo

# Nombre de archivo unico por corrida (finde + fecha de corrida) - nunca
# sobreescribe una corrida anterior, ni de otro finde ni de este mismo.
RUTA_SALIDA = strategy.generar_ruta_salida(WEEKEND_INICIO, WEEKEND_FIN, carpeta="../data/output")
print(f"Se va a guardar en: {RUTA_SALIDA}")

## Conexion a Snowflake

SSO interactivo (`authenticator="externalbrowser"`) - al ejecutar la celda
se abre el navegador para el login de Google.

In [ ]:
conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT CURRENT_USER(), CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_DATABASE()")
print(cur.fetchone())

## Construir la estrategia

Une todos los pasos (exclusion comercial, catalogo real, elegibilidad,
Tema Mundial, descuento maximo, elasticidad real + fallback, score,
cascada de mecanica, guardrail economico, confianza + backtesting, dia de
ejecucion, escenarios) y exporta el Excel final de una sola vez. El detalle
de cada paso esta documentado como docstring en `backend/pricing/strategy.py`.

In [ ]:
if postmortem.plan_aun_no_ejecutado(WEEKEND_FIN):
    df = strategy.construir_estrategia(
        cur=cur,
        ruta_oportunidad=RUTA_OPORTUNIDAD,
        ruta_descuentos=RUTA_DESCUENTOS,
        ruta_salida=RUTA_SALIDA,
        weekend_inicio=WEEKEND_INICIO,
        weekend_fin=WEEKEND_FIN,
    )
else:
    df = None
    print("WEEKEND_FIN ya paso - no se construye una estrategia nueva para una ventana ya ejecutada.")
    print("Ver mas abajo la seccion de post-mortem para medir/subir lo que realmente se ejecuto.")

## Resultados finales

In [ ]:
if df is None:
    print("No se construyo una estrategia nueva (WEEKEND_FIN ya paso) - nada que reportar aqui.")
else:
    ofer = df[df["MECANICA"] != "Sin oferta"]

    print("=== ESTRATEGIA ===")
    print(df["MECANICA"].value_counts().to_string())
    print(f"\nOfertas: {len(ofer)} de {len(df)} ({len(ofer)/len(df)*100:.1f}%) | "
          f"descuento exhibido promedio: {ofer['DESC_EFECTIVO'].mean()*100:.1f}% | "
          f"margen minimo en promocion: {ofer['MARGEN_OFERTA'].min():.2f}%")
    print(f"SKUs que requieren aprobacion Comercial (no incluidos, d_max real > 30%): "
          f"{df['REQUIERE_APROBACION'].sum()}")

    print("\n=== PROYECCION ESCENARIO BASE ===")
    print(f"Unidades: {ofer['Q0_DIA'].sum():,.0f} -> {ofer['Q1_DIA'].sum():,.0f} "
          f"({(ofer['Q1_DIA'].sum()/ofer['Q0_DIA'].sum()-1)*100:+.1f}%)")
    print(f"GMV:      ${ofer['GMV_BASE_DIA'].sum():,.0f} -> ${ofer['GMV_PROY_DIA'].sum():,.0f} "
          f"({(ofer['GMV_PROY_DIA'].sum()/ofer['GMV_BASE_DIA'].sum()-1)*100:+.1f}%)")
    print(f"Utilidad: ${ofer['UTIL_BASE_DIA'].sum():,.0f} -> ${ofer['UTIL_PROY_DIA'].sum():,.0f} "
          f"({(ofer['UTIL_PROY_DIA'].sum()/ofer['UTIL_BASE_DIA'].sum()-1)*100:+.1f}%)")

    print("\n=== POR DIA ===")
    print(ofer.groupby("DIA_EJECUCION").agg(Ofertas=("SKU", "count"),
          GMV_inc=("GMV_INC_DIA", "sum"), Util_inc=("UTIL_INC_DIA", "sum")).round(0).to_string())

    print("\n=== CONFIANZA DE LA PROYECCION (ofertas finales) ===")
    print(ofer["CONFIANZA_PROYECCION"].value_counts())

    mun = ofer[ofer["TEMA_MUNDIAL"]]
    print(f"\n=== MUNDIAL ===\nOfertas: {len(mun)} | GMV incremental: ${mun['GMV_INC_DIA'].sum():,.0f} | "
          f"Utilidad incremental: ${mun['UTIL_INC_DIA'].sum():,.0f}")

## Subir el plan a Snowflake

Dos momentos distintos, con reglas distintas:

1. **Construir la estrategia para el proximo finde** (el que no ha pasado
   todavia): la celda de abajo sube el plan automaticamente, sobreescribiendo
   cualquier intento anterior para esa misma ventana - solo importa la
   ultima version, porque nada se ha ejecutado aun.
2. **Medir una promo que ya paso**: la celda de arriba NO sube nada
   automatico (para no pisar el plan real con una reconstruccion de hoy).
   Hay que subir manualmente el Excel que de verdad se ejecuto - ver la
   celda "Subir manualmente" mas abajo.

Las vistas (`WKND_PROMO_RESULTS_V`, `WKND_PLAN_VS_ACTUAL_V`, `WKND_POSTMORTEM_PROMO_V`)
se crean/reemplazan cada vez que se sube un plan - son recurrentes, no hay
que tocarlas aparte.

In [ ]:
# 1. Auto-subida: solo si WEEKEND_FIN todavia no paso (plan de un finde futuro)
if postmortem.plan_aun_no_ejecutado(WEEKEND_FIN):
    estrategia_df = df[df["MECANICA"] != "Sin oferta"].rename(columns={"MARGEN_OFERTA": "MARGEN_OFERTA_%"})
    n_subidas = postmortem.subir_plan_y_crear_vistas(cur, estrategia_df, WEEKEND_INICIO, WEEKEND_FIN)
    print(f"Plan subido a SANDBOX.WKND_PROMO_PLAN: {n_subidas} filas "
          f"(finde aun no ejecutado, se sobreescribe en cada corrida)")
else:
    print("WEEKEND_FIN ya paso - no se subio nada automatico para no pisar el plan real ejecutado.")
    print('Usa la celda "Subir manualmente el plan real de una promo ya pasada" de abajo.')

### Subir manualmente el plan real de una promo ya pasada

Si `WEEKEND_FIN` ya paso, esta celda busca en disco (por fecha de finde,
con `strategy.buscar_salida_historica`) el Excel que realmente se ejecuto y
lo sube a `WKND_PROMO_PLAN` - es la version que "gano" esa semana, no una
reconstruccion de hoy. Si `WEEKEND_FIN` no ha pasado, no hace nada (ya se
subio arriba, automatico).

In [ ]:
if not postmortem.plan_aun_no_ejecutado(WEEKEND_FIN):
    try:
        ruta_historica = strategy.buscar_salida_historica(WEEKEND_INICIO, WEEKEND_FIN, carpeta="../data/output")
    except FileNotFoundError as e:
        print(e)
        print("No se sube nada - no existe un Excel para esta ventana.")
    else:
        print(f"Plan leido de: {ruta_historica}")
        estrategia_df_historica = pd.read_excel(ruta_historica, sheet_name="Estrategia FDS")
        n_subidas = postmortem.subir_plan_y_crear_vistas(cur, estrategia_df_historica, WEEKEND_INICIO, WEEKEND_FIN)
        print(f"Plan subido a SANDBOX.WKND_PROMO_PLAN: {n_subidas} filas")
else:
    print("WEEKEND_FIN todavia no paso - el plan ya se subio automaticamente arriba, no hace falta esta celda.")

### Medir una promo ya pasada

Cambia `PROMO_INICIO`/`PROMO_FIN` a la ventana que quieras medir (ej. el
FDS 3-5 jul que ya corrio) y correlo - responde que mecanica gano, separado
por si fue nuestra propuesta u otra promo que coincidio en la ventana.

In [ ]:
PROMO_INICIO = pd.Timestamp("2026-07-03")
PROMO_FIN    = pd.Timestamp("2026-07-05")

resumen_mecanica = postmortem.performance_por_mecanica(cur, PROMO_INICIO, PROMO_FIN)
resumen_mecanica

### Validar la tasa de redencion real (FACT_FULFILLMENT_LINE)

`MX_JUSTO_PROD.DM_CORE.FACT_FULFILLMENT_LINE` trae detalle a nivel de linea
de pedido con `IS_DISCOUNT`/`IS_BULK_APPLIED` - permite comparar la tasa de
redencion **real** contra los supuestos declarados en
`strategy.REDENCION` (35%/40%/55%). Si la diferencia es grande, vale la pena
recalibrar esos supuestos igual que se hizo con `MULT_PROMO` en el
backtesting de arriba.

In [ ]:
redencion_real = postmortem.validar_redencion_real(cur, PROMO_INICIO, PROMO_FIN)
redencion_real